<a href="https://colab.research.google.com/github/giuliobarde/web_data_mining_project/blob/main/Mini_Project_2_RetrieveFromPinecone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 4.4 MB/s eta 0:00:00


In [3]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 4.9 MB/s eta 0:00:00


In [12]:
import json
import boto3
import pandas as pd
from typing import List, Dict, Any, Optional
import pinecone
import time
from botocore.config import Config
from botocore import UNSIGNED
import random
import requests

class NewsProcessor:
    """
    A class for processing news articles from S3 and storing them in Pinecone vector database
    """

    def __init__(self, api_key: str, index_name: str, namespace: str, category: str):
        """
        Initialize the NewsProcessor with Pinecone credentials and team information.

        Args:
            api_key: Pinecone API key
            index_name: Pinecone index name
            team_name: Team name
            category: Assigned category
        """
        self.api_key = api_key
        self.index_name = index_name
        self.namespace = namespace
        self.category = category
        self.index = None

    def connect_to_pinecone(self) -> None:
        """Connect to Pinecone and initialize the index."""
        try:
            pinecone_client = pinecone.Pinecone(api_key=self.api_key, environment="us-east-1")
            self.index = pinecone_client.Index(self.index_name)
            print(f"Connected to Pinecone index: {self.index_name}")
        except Exception as e:
            print(f"Error connecting to Pinecone: {str(e)}")
            print("Please check your API key, index name, and environment.")
            raise

    def fetch_from_s3(self, bucket_name: str, team_folder: str, category: str = None,
                      aws_access_key: str = None, aws_secret_key: str = None) -> List[Dict]:
        """
        Fetch news articles from S3 bucket.

        Args:
            bucket_name: S3 bucket name
            team_folder: Team folder name
            category: Category to filter sources
            aws_access_key: AWS access key
            aws_secret_key: AWS secret key

        Returns:
            List of articles combined from all relevant source files.
        """
        # Initialize S3 client.
        if aws_access_key and aws_secret_key:
            s3_client = boto3.client(
                's3',
                aws_access_key_id=aws_access_key,
                aws_secret_access_key=aws_secret_key
            )
        else:
            s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))

        try:
            # List all objects in the sources directory.
            sources_prefix = f"{team_folder}sources/"
            print(f"Listing objects in s3://{bucket_name}/{sources_prefix}")
            response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=sources_prefix)

            if 'Contents' not in response:
                print(f"No source files found in s3://{bucket_name}/{sources_prefix}")
                return []

            # Filter sources based on category
            source_files = []
            for obj in response['Contents']:
                file_key = obj['Key']
                if not file_key.endswith('.json'):
                    continue

                if category:
                    source_name = file_key.split('/')[-1].replace('.json', '')
                    if category.lower() in source_name.lower():
                        source_files.append(file_key)
                else:
                    source_files.append(file_key)

            print(f"Found {len(source_files)} source files")

            # Collect articles from all matching source files.
            all_articles = []
            for file_key in source_files:
                try:
                    print(f"Fetching {file_key}")
                    response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
                    file_content = response['Body'].read().decode('utf-8')
                    source_data = json.loads(file_content)

                    for article_id, article in source_data.items():
                        article['source_file'] = file_key.split('/')[-1]
                        all_articles.append(article)

                    print(f"Added {len(source_data)} articles from {file_key}")
                except Exception as e:
                    print(f"Error processing {file_key}: {str(e)}")

            print(f"Total articles collected: {len(all_articles)}")
            return all_articles

        except Exception as e:
            print(f"Error fetching from S3: {str(e)}")
            if "AccessDenied" in str(e):
                print("Access denied. Check your IAM permissions or AWS credentials.")
            elif "NoSuchBucket" in str(e):
                print(f"Bucket '{bucket_name}' not found. Check the bucket name.")
            raise

    def process_articles(self, articles: List[Dict]) -> List[Dict]:
        """
        Process articles to prepare them for vector storage.
        """
        processed_articles = []
        for i, article in enumerate(articles):
            title = article.get('title')
            description = article.get('description')
            content = article.get('content')
            url = article.get('url')
            published_at = article.get('publishedAt')
            source_name = article.get('source', {}).get('name', '')
            source_file = article.get('source_file')
            author = article.get('author')

            text_content = f"Title: {title}\nDescription: {description}\nContent: {content}"

            processed_article = {
                'id': f"{self.namespace}_{self.category}_{source_name}_{i}",
                'text': text_content,
                'metadata': {
                    'team': self.namespace,
                    'category': self.category,
                    'title': title,
                    'url': url,
                    'published_at': published_at,
                    'source': source_name,
                    'source_file': source_file,
                    'author': author if author is not None else "Unknown"
                }
            }
            processed_articles.append(processed_article)
        return processed_articles

    def upload_to_pinecone(self, processed_articles: List[Dict]) -> None:
        """
        Upload processed articles to Pinecone.
        """
        if not self.index:
            raise ValueError("Not connected to Pinecone. Call connect_to_pinecone() first.")

        # Prepare records for upsert
        upsert_batch = []
        for article in processed_articles:
            random.seed(42)
            vector = [random.uniform(-0.01, 0.01) for _ in range(1024)]

            upsert_record = {
                'id': article['id'],
                'values': vector,
                'metadata': {
                    **article['metadata'],
                    'text': article['text']
                }
            }
            upsert_batch.append(upsert_record)

        batch_size = 10
        successful_uploads = 0
        for i in range(0, len(upsert_batch), batch_size):
            batch = upsert_batch[i:i+batch_size]
            try:
                self.index.upsert(vectors=batch, namespace=self.namespace)
                successful_uploads += len(batch)
                print(f"Uploaded batch {i//batch_size + 1}/{(len(upsert_batch)-1)//batch_size + 1}")
            except Exception as e:
                print(f"Error uploading batch {i//batch_size + 1}: {str(e)}")
                for j, record in enumerate(batch):
                    try:
                        self.index.upsert(vectors=[record])
                        successful_uploads += 1
                        print(f"  Successfully uploaded record {i+j+1}")
                    except Exception as e2:
                        print(f"  Failed to upload record {i+j+1}: {str(e2)}")

        print(f"Upload complete. Successfully uploaded {successful_uploads} out of {len(processed_articles)} articles.")

    def query_pinecone(self, query_text: str, top_k: int = 5, filter_params: Dict = None) -> List[Dict]:
        """
        Query Pinecone for similar articles.
        """
        if not self.index:
            raise ValueError("Not connected to Pinecone. Call connect_to_pinecone() first.")

        random.seed(42)
        vector = [random.uniform(-0.01, 0.01) for _ in range(1024)]

        query_params = {
            'top_k': top_k,
            'include_metadata': True,
        }
        if filter_params:
            query_params['filter'] = filter_params

        try:
            results = self.index.query(vector=vector, **query_params)
            return results.matches if hasattr(results, 'matches') else []
        except Exception as e:
            print(f"Error querying Pinecone: {str(e)}")
            return []

    def analyze_category_insights(self) -> Dict:
        """
        Analyze category-specific insights from stored articles.
        """
        filter_params = {
            'team': self.namespace,
            'category': self.category
        }

        category_queries = {
            'Health': ["medical research", "disease prevention", "healthcare policy", "wellness trends"],
            'Finance': ["stock market", "investment trends", "economic forecast", "financial news"],
            'Politics': ["election", "government policy", "political campaign", "international relations"],
            'Policies': ["regulation", "legislation", "policy reform", "government initiatives"],
            'World News': ["international events", "global crisis", "diplomatic relations", "world economy"],
            'Investment': ["stock performance", "investment strategy", "market analysis", "portfolio management"],
            'Leisure': ["entertainment", "sports events", "travel destinations", "recreational activities"]
        }
        queries = category_queries.get(self.category, ["trending topics", "recent developments", "major events"])

        insights = {}
        for query in queries:
            results = self.query_pinecone(query, top_k=3, filter_params=filter_params)
            insights[query] = results
        return insights

if __name__ == "__main__":
    processor = NewsProcessor(
        api_key="pcsk_6akU8Z_2BXXXDSBKbvFCn4sciNM2FeJC6PwAt6wFwQeQjoJKDSjysRbtyBAdUfRv6z87e6",
        index_name="cus635",
        namespace="Team_1",
        category="Finance"
    )

    # Connect to Pinecone
    processor.connect_to_pinecone()

    bucket_name = "cus635-spring2025"
    team_folder = "TEAM_1/"

    try:
        s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
        print(f"Checking available folders in bucket: {bucket_name}")
        response = s3.list_objects_v2(Bucket=bucket_name, Delimiter="/")
        if 'CommonPrefixes' in response:
            print("Available team folders:")
            for prefix in response['CommonPrefixes']:
                print(f"- {prefix['Prefix']}")
        else:
            print("No team folders found. Checking all files in bucket...")
            response = s3.list_objects_v2(Bucket=bucket_name)
            if 'Contents' in response:
                for obj in response['Contents'][:10]:
                    print(f"- {obj['Key']}")
            else:
                print("No files found in bucket.")

        print(f"\nFetching articles from s3://{bucket_name}/{team_folder}")
        articles = processor.fetch_from_s3(bucket_name=bucket_name, team_folder=team_folder)
        if not articles:
            print("\nNo articles found in the default team folder. Trying TEAM_2/...")
            team_folder = "TEAM_2/"
            articles = processor.fetch_from_s3(bucket_name=bucket_name, team_folder=team_folder)

        if not articles:
            print("\nStill no articles found. Listing available files in the bucket for reference:")
            response = s3.list_objects_v2(Bucket=bucket_name)
            if 'Contents' in response:
                for obj in response['Contents'][:20]:
                    print(f"- {obj['Key']}")
            else:
                print("No files found in bucket. Please verify the bucket name and permissions.")
        else:
            processed_articles = processor.process_articles(articles)
            print(f"Processed {len(processed_articles)} articles, uploading to Pinecone...")
            processor.upload_to_pinecone(processed_articles)
            print("Upload complete!")

            print("\nPerforming sample query:")
            category_queries = {
                "Finance": ["stock market", "investment trends", "financial news"],
                "Health": ["medical research", "healthcare policy", "wellness trends"],
                "Politics": ["election", "government policy", "political debate"],
                "Policies": ["regulation", "legislation", "policy reform"],
                "World News": ["international relations", "global economy", "world events"],
                "Investment": ["stock market", "investment strategy", "portfolio management"],
                "Leisure": ["entertainment", "travel", "sports events"]
            }

            relevant_queries = category_queries.get(processor.category, ["trending topics"])
            for query in relevant_queries:
                print(f"\nQuerying for: '{query}'")
                query_results = processor.query_pinecone(query, top_k=3)
                for i, result in enumerate(query_results, 1):
                    print(f"{i}. {result.metadata.get('title')} (Score: {result.score:.4f})")
                    print(f"   Source: {result.metadata.get('source')}")

            print("\nAnalyzing category insights:")
            insights = processor.analyze_category_insights()
            for query, results in insights.items():
                print(f"\nInsights for '{query}':")
                for i, result in enumerate(results, 1):
                    print(f"{i}. {result.metadata.get('title')} (Score: {result.score:.4f})")

    except Exception as e:
        print(f"Error: {str(e)}")
        print("If you're facing S3 access issues, ensure your AWS credentials are configured correctly.")


Connected to Pinecone index: cus635
Checking available folders in bucket: cus635-spring2025
Available team folders:
- TEAM_1/
- TEAM_2/
- TEAM_3/
- TEAM_4/
- TEAM_5/
- TEAM_6/
- TEAM_7/
- TEAM_8/

Fetching articles from s3://cus635-spring2025/TEAM_1/
Listing objects in s3://cus635-spring2025/TEAM_1/sources/
Found 58 source files
Fetching TEAM_1/sources/ABC_News.json
Added 9 articles from TEAM_1/sources/ABC_News.json
Fetching TEAM_1/sources/ABC_News_AU_.json
Added 1 articles from TEAM_1/sources/ABC_News_AU_.json
Fetching TEAM_1/sources/ANSA_it.json
Added 1 articles from TEAM_1/sources/ANSA_it.json
Fetching TEAM_1/sources/AppleInsider.json
Added 1 articles from TEAM_1/sources/AppleInsider.json
Fetching TEAM_1/sources/Associated_Press.json
Added 2 articles from TEAM_1/sources/Associated_Press.json
Fetching TEAM_1/sources/BBC_News.json
Added 4 articles from TEAM_1/sources/BBC_News.json
Fetching TEAM_1/sources/BBC_Sport.json
Added 4 articles from TEAM_1/sources/BBC_Sport.json
Fetching TEAM_